# Test pipeline Phase 1 + Phase 2 sur ais_data.csv

Adapte et fait tourner Phase 1 (HDBSCAN + features) et Phase 2 (LBO + gravity score)
sur `data/ais_data.csv` — Rotterdam/Manche, ~17 min, 2075 navires.

Stratégie : découper en fenêtres de 2 min → pseudo-jours.

In [ ]:
import sys
from pathlib import Path

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

import polars as pl
import numpy as np
from datetime import datetime, timedelta

df_raw = pl.read_csv(ROOT / 'data' / 'ais_data.csv')
print(f'Donnees brutes : {df_raw.shape}')
print(df_raw.head(3))


In [ ]:
# Normalisation colonnes -> schema Marine Cadastre
df = (
    df_raw
    .rename({'mmsi': 'MMSI', 'lat': 'LAT', 'lon': 'LON', 'speed': 'SOG', 'course': 'COG'})
    .with_columns(
        pl.from_epoch(pl.col('timestamp').cast(pl.Int64), time_unit='s').alias('BaseDateTime'),
        pl.lit(None).cast(pl.Float64).alias('Heading'),
        pl.lit(None).cast(pl.Float64).alias('Draft'),
        pl.lit(0).cast(pl.Int64).alias('VesselType'),
    )
    .filter(
        (pl.col('MMSI').cast(pl.Int64) >= 200_000_000) &
        (pl.col('MMSI').cast(pl.Int64) <= 999_999_999) &
        (pl.col('SOG') >= 0) & (pl.col('SOG') <= 50)
    )
)
print(f'Apres filtre : {df.shape} | {df["MMSI"].n_unique()} navires')


In [ ]:
# Decoupage en fenetres de 2 min (pseudo-jours)
WINDOW_MINUTES = 2

t_start = df['BaseDateTime'].min()
df = df.with_columns(
    ((pl.col('BaseDateTime') - t_start).dt.total_seconds() // (WINDOW_MINUTES * 60))
    .cast(pl.Int32).alias('window_id')
)

windows = sorted(df['window_id'].unique().to_list())
print(f'Fenetres de {WINDOW_MINUTES} min : {len(windows)} pseudo-jours')
for w in windows:
    wdf = df.filter(pl.col('window_id') == w)
    print(f'  Fenetre {w:2d} : {len(wdf):4d} msgs | {wdf["MMSI"].n_unique():4d} navires')


In [ ]:
# Phase 1 : kinematic filter + HDBSCAN + features par fenetre
from src.ingestion.kinematic_filter import prepare_kinematics
from src.clustering.hdbscan_daily import cluster_day_from_df
from src.clustering.features_daily import compute_daily_features

all_features = []
# Chaque fenetre = 1 pseudo-jour distinct (date incrementee de 1 jour par fenetre)
fake_date_base = datetime(2026, 1, 1).date()

for w in windows:
    wdf = df.filter(pl.col('window_id') == w).drop('window_id')
    fake_date = fake_date_base + timedelta(days=w)  # date unique par fenetre
    try:
        prepared = prepare_kinematics(wdf)
        cluster_df, prepared_df = cluster_day_from_df(prepared)
        if cluster_df is None:
            print(f'  Fenetre {w}: pas assez de clusters')
            continue
        feat = compute_daily_features(prepared_df, cluster_df, fake_date)
        if feat:
            feat['date'] = fake_date.isoformat()
            feat['window_id'] = w
            all_features.append(feat)
            print(f"  Fenetre {w} ({fake_date}) ok  rho={feat.get('utilization_rate_rho',0):.3f}  "
                  f"clusters={feat.get('hdbscan_cluster_count',0)}  "
                  f"vessels={feat.get('vessel_count',0)}")
        else:
            print(f'  Fenetre {w}: features vides')
    except Exception as e:
        import traceback; traceback.print_exc()
        print(f'  Fenetre {w}: ERREUR - {e}')

print(f'\nTotal : {len(all_features)} fenetres avec features')


In [ ]:
# Resultats Phase 1
if not all_features:
    print('Aucune feature calculee')
else:
    df_feat = pl.DataFrame(all_features)
    print(f'Shape : {df_feat.shape}')
    cols = [c for c in ['window_id', 'vessel_count', 'SOG_mean',
                        'utilization_rate_rho', 'hdbscan_cluster_count',
                        'hdbscan_noise_ratio', 'membership_score_mean']
            if c in df_feat.columns]
    print(df_feat.select(cols))


In [ ]:
# Phase 2 : LBO Manifold
if len(all_features) < 5:
    print(f'Trop peu de fenetres ({len(all_features)}) - min 5 requis')
else:
    from src.manifold.lbo import run_lbo

    FEATURE_COLS = [
        'vessel_count', 'SOG_mean', 'SOG_std', 'SOG_median',
        'utilization_rate_rho', 'hdbscan_cluster_count', 'hdbscan_noise_ratio',
        'membership_score_mean', 'membership_score_std',
        'draft_mean', 'draft_std', 'blocked_capacity', 'tanker_ratio',
    ]

    # Colonnes absentes de ais_data.csv -> imputer a 0
    missing = [c for c in FEATURE_COLS if c not in df_feat.columns]
    present = [c for c in FEATURE_COLS if c in df_feat.columns]
    print(f'Colonnes presentes : {present}')
    print(f'Colonnes imputees a 0 : {missing}')

    df_feat_filled = df_feat
    for c in missing:
        df_feat_filled = df_feat_filled.with_columns(pl.lit(0.0).alias(c))

    tmp_path = ROOT / 'data' / 'features' / '_test_features.parquet'
    tmp_path.parent.mkdir(parents=True, exist_ok=True)
    df_feat_lbo = df_feat_filled.with_columns(pl.col('date').str.to_date())
    df_feat_lbo.write_parquet(tmp_path)

    k = min(5, len(all_features) - 1)
    n_eig = min(4, len(all_features) - 2)
    print(f'LBO : k={k}, n_eigenvectors={n_eig}, n_points={len(all_features)}')

    try:
        df_manifold = run_lbo(features_path=tmp_path, k=k, n_eigenvectors=n_eig)
        print(f'\nManifold OK : {df_manifold.shape}')
        phi_cols = [c for c in df_manifold.columns if c.startswith('phi')]
        print(df_manifold.select(['date'] + phi_cols + ['is_characteristic']))
    except Exception as e:
        import traceback; traceback.print_exc()


In [ ]:
# Gravity Score
if len(all_features) >= 5 and 'df_manifold' in dir():
    try:
        from src.manifold.gravity_score import compute_gravity_score

        dates = df_feat_lbo['date'].sort().to_list()
        mid = len(dates) // 2
        # Simuler un 'evenement' sur les 2-3 fenetres centrales
        event_start = dates[mid]
        event_end   = dates[min(mid + 2, len(dates) - 1)]
        print(f'Fenetre evenement simulee : {event_start} -> {event_end}')
        print(f'Baseline : {len([d for d in dates if d < event_start])} fenetres avant evenement')

        df_gravity = compute_gravity_score(
            df_manifold, harvey_start=event_start, harvey_end=event_end
        )
        print(df_gravity.select(['date', 'gravity_score', 'deviation_score'])
              .sort('gravity_score', descending=True))
    except Exception as e:
        import traceback; traceback.print_exc()


In [ ]:
# Carte Folium — clusters HDBSCAN (derniere fenetre)
import folium

last_w = max(all_features, key=lambda f: f['window_id'])['window_id']
last_df = df.filter(pl.col('window_id') == last_w).drop('window_id')
last_prepared = prepare_kinematics(last_df)
last_cluster_df, last_prepared_df = cluster_day_from_df(last_prepared)

center = [float(last_prepared_df['LAT'].mean()), float(last_prepared_df['LON'].mean())]
m = folium.Map(location=center, zoom_start=10, tiles='CartoDB positron')

sample = last_prepared_df.sample(min(300, len(last_prepared_df)))
for row in sample.iter_rows(named=True):
    folium.CircleMarker(
        location=[row['LAT'], row['LON']],
        radius=2, color='lightgray', fill=True, fill_opacity=0.4,
    ).add_to(m)

if last_cluster_df is not None:
    for row in last_cluster_df.iter_rows(named=True):
        ctype = row.get('cluster_type', '?')
        color = 'blue' if ctype == 'docked' else 'orange'
        lat_key = 'LAT_mean' if 'LAT_mean' in row else 'LAT'
        lon_key = 'LON_mean' if 'LON_mean' in row else 'LON'
        folium.CircleMarker(
            location=[row[lat_key], row[lon_key]],
            radius=8, color=color, fill=True, fill_opacity=0.8,
            tooltip=f"Cluster {row.get('cluster_label')} | {ctype}",
        ).add_to(m)

n_clusters = len(last_cluster_df) if last_cluster_df is not None else 0
print(f'Carte : {n_clusters} clusters | {len(last_prepared_df)} navires')
m
